In [1]:
import numpy as np
import math

In [2]:
class Node_A():
    
    def __init__(self, state, parent, action, g, h):
        self.state = state
        self.parent = parent
        self.action = action
        self.g = g
        self.h = h

In [3]:
def create_mat():
    
    new_mat = np.array([
        ['1', '1', '1', '0'],
        ['0', '0', 'x', '1'],
        ['1', '1', '0', '0'],
        ['1', '1', '0', '1']
    ])
    
    row, col = find_vacuum(new_mat)
    
    return new_mat, row, col

In [4]:
def find_vacuum(matrix):
    pos = np.argwhere(matrix == 'x')[0]
    
    return pos[0], pos[1]

In [5]:
def moves(matrix):
    
    lst_move = []
    row, col = find_vacuum(matrix)
    
    directions = [
        ['LEFT', 0, -1],
        ['RIGHT', 0, 1],
        ['UP', -1, 0],
        ['DOWN', 1, 0]
    ]
    
    for dir, x, y in directions:
        row_new = row + x
        col_new = col + y
        
        if row_new < 4 and row_new >= 0 and col_new < 4 and col_new >= 0:
            lst_move.append([dir, row_new, col_new])
            
    return lst_move

In [6]:
def solution(node):
    path = []
    
    while node.parent is not None:
        path.append((node.action, node.state))
        
        node = node.parent
    
    path.reverse()
    
    return path

In [7]:
def distance_to_dirty_cell(matrix, row, col):
    dis = 0
    for i in range(0, 4):
        for j in range(0, 4):
            if matrix[i, j] == '1':
                dis += abs(row - i) + abs(col - j)
    
    return dis

In [8]:
def count_dirty_position(matrix):
    count = 0
    for x in range(4):
        for y in range(4):
            if matrix[x][y] == '1':
                count += 1
    return count

In [11]:
def ida_star(matrix, row, col):

    print("TRANG THAI BAN DAU")
    print(matrix)

    g = count_dirty_position(matrix)
    h = distance_to_dirty_cell(matrix, row, col)

    cost = g + h

    while True:

        result, reached, next_cost = depth_limited_search_ida(matrix, row, col, cost)

        if result is not None:
            return result, reached

        if next_cost == math.inf:

            print("KHONG TIM THAY LOI GIAI")
            return None, reached

        cost = next_cost

        print(f"TANG cost LEN: {cost}")

In [ ]:
def depth_limited_search_ida(matrix, row, col, cost):

    matrix = matrix.copy()

    matrix[row, col] = 'x'

    g = count_dirty_position(matrix)
    h = distance_to_dirty_cell(matrix, row, col)

    node = Node_A(matrix, None, None, g, h)

    frontier = [node]
    frontier_state = {}
    temp_start = tuple(map(tuple, node.state))
    frontier_state[temp_start] = g + h
    reached = {}
    next_cost = math.inf

    while len(frontier) != 0:

        node = frontier.pop(len(frontier) - 1)

        temp_mat = tuple(map(tuple, node.state))

        frontier_state.pop(temp_mat, None)

        f = node.g + node.h

        if f > cost:

            next_cost = min(next_cost, f)
            continue

        reached[temp_mat] = f

        if '1' not in node.state:
            solutions = solution(node)
            print("TIM THAY LOI GIAI !!!")
            for action, state in solutions:
                print("HUONG DI:", action)
                print(state)
            return solutions, reached, next_cost
        actions = moves(node.state)
        row, col = find_vacuum(node.state)

        for dir, x, y in actions:

            new_node = node.state.copy()
            new_node[row, col], new_node[x, y] = '0', 'x'
            temp_node = tuple(map(tuple, new_node))

            g_new = count_dirty_position(new_node)
            h_new = distance_to_dirty_cell(new_node, x, y)

            f_new = g_new + h_new

            if temp_node in reached:
                if f_new >= reached[temp_node]:
                    continue

                else:
                    reached[temp_node] = f_new

            if temp_node in frontier_state:

                if f_new < frontier_state[temp_node]:
                    for i in range(len(frontier)):
                        state = tuple(map(tuple, frontier[i].state))
                        if state == temp_node:

                            frontier[i].g = g_new
                            frontier[i].h = h_new
                            frontier[i].parent = node
                            frontier[i].action = dir

                            frontier_state[temp_node] = f_new
                            break
                else:
                    continue

            if f_new > cost:
                next_cost = min(next_cost, f_new)
                continue

            if temp_node not in frontier_state and temp_node not in reached:
                child = Node_A(new_node, node, dir, g_new, h_new)
                frontier.append(child)
                frontier_state[temp_node] = f_new

    return None, reached, next_cost

In [12]:
if __name__ == "__main__":
    mat, row, col = create_mat()
    ida_star(mat, row, col)

TRANG THAI BAN DAU
[['1' '1' '1' '0']
 ['0' '0' 'x' '1']
 ['1' '1' '0' '0']
 ['1' '1' '0' '1']]
TIM THAY LOI GIAI !!!
HUONG DI: DOWN
[['1' '1' '1' '0']
 ['0' '0' '0' '1']
 ['1' '1' 'x' '0']
 ['1' '1' '0' '1']]
HUONG DI: LEFT
[['1' '1' '1' '0']
 ['0' '0' '0' '1']
 ['1' 'x' '0' '0']
 ['1' '1' '0' '1']]
HUONG DI: DOWN
[['1' '1' '1' '0']
 ['0' '0' '0' '1']
 ['1' '0' '0' '0']
 ['1' 'x' '0' '1']]
HUONG DI: UP
[['1' '1' '1' '0']
 ['0' '0' '0' '1']
 ['1' 'x' '0' '0']
 ['1' '0' '0' '1']]
HUONG DI: UP
[['1' '1' '1' '0']
 ['0' 'x' '0' '1']
 ['1' '0' '0' '0']
 ['1' '0' '0' '1']]
HUONG DI: UP
[['1' 'x' '1' '0']
 ['0' '0' '0' '1']
 ['1' '0' '0' '0']
 ['1' '0' '0' '1']]
HUONG DI: DOWN
[['1' '0' '1' '0']
 ['0' 'x' '0' '1']
 ['1' '0' '0' '0']
 ['1' '0' '0' '1']]
HUONG DI: DOWN
[['1' '0' '1' '0']
 ['0' '0' '0' '1']
 ['1' 'x' '0' '0']
 ['1' '0' '0' '1']]
HUONG DI: DOWN
[['1' '0' '1' '0']
 ['0' '0' '0' '1']
 ['1' '0' '0' '0']
 ['1' 'x' '0' '1']]
HUONG DI: RIGHT
[['1' '0' '1' '0']
 ['0' '0' '0' '1']
 ['1' 